# 02: Federal Contract Schedule & Vendor Analysis

**Goal:** Analyze federal IT contract patterns from USASpending.gov to understand award distributions, vendor concentration, and schedule risk patterns.

**Data:** USASpending.gov — 1,000 federal contracts ($169.9B total obligation), 56 IT-specific contracts analyzed

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

## Load Data

In [ ]:
df = pd.read_csv('../data/federal_contracts_all.csv')
df['award_amount'] = pd.to_numeric(df['award_amount'], errors='coerce')
df['duration_days'] = pd.to_numeric(df['duration_days'], errors='coerce')
df['start_date'] = pd.to_datetime(df['start_date'], errors='coerce')
df['end_date'] = pd.to_datetime(df['end_date'], errors='coerce')

print(f"Total contracts: {len(df)}")
print(f"Agencies: {df['agency'].nunique()}")
print(f"Total obligation: ${df['award_amount'].sum()/1e9:.1f}B")
print(f"Date range: {df['start_date'].min().date()} to {df['end_date'].max().date()}")
df.head()

## Contract Duration Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
df['duration_days'].dropna().hist(bins=30, ax=axes[0], color='#3498db', edgecolor='white')
axes[0].axvline(df['duration_days'].mean(), color='red', linestyle='--',
                label=f"Mean: {df['duration_days'].mean():.0f}d")
axes[0].set_title('Contract Duration Distribution')
axes[0].set_xlabel('Days')
axes[0].legend()

# By agency (top 8)
top_agencies = df.groupby('agency')['award_amount'].sum().nlargest(8).index
subset = df[df['agency'].isin(top_agencies)]
subset.boxplot(column='duration_days', by='agency', ax=axes[1], rot=45)
axes[1].set_title('Duration by Top Agencies')
axes[1].set_xlabel('')

plt.tight_layout()
plt.show()

## Agency-Level Award Analysis

In [ ]:
agency_stats = df.groupby('agency').agg({
    'award_amount': ['sum', 'mean', 'count'],
    'duration_days': ['mean', 'std']
}).round(2)
agency_stats.columns = ['_'.join(col) for col in agency_stats.columns]
agency_stats = agency_stats.sort_values('award_amount_sum', ascending=False)

print(agency_stats.head(10))

# Plot top 10
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
top10 = agency_stats.head(10)

top10['award_amount_sum'].plot(kind='barh', ax=axes[0], color='#2ecc71')
axes[0].set_title('Total Obligation by Agency')
axes[0].set_xlabel('USD')

top10['duration_days_mean'].plot(kind='barh', ax=axes[1], color='#e67e22')
axes[1].set_title('Mean Contract Duration by Agency')
axes[1].set_xlabel('Days')

plt.tight_layout()
plt.show()

## Vendor Concentration (HHI)

In [ ]:
recipient_share = df.groupby('recipient')['award_amount'].sum()
recipient_share = recipient_share / recipient_share.sum()
hhi = (recipient_share ** 2).sum() * 10000

print(f"Vendor Concentration (HHI): {hhi:.0f}")
print(f"Top 10 vendors: {recipient_share.nlargest(10).sum()*100:.1f}% of total")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

recipient_share.nlargest(15).plot(kind='barh', ax=axes[0], color='#9b59b6')
axes[0].set_title('Top 15 Vendors by Award Share')
axes[0].set_xlabel('Share of Total Obligation')

recipient_share.hist(bins=30, ax=axes[1], color='#9b59b6', edgecolor='white')
axes[1].set_title('Vendor Share Distribution')
axes[1].set_xlabel('Share of Total Obligation')

plt.tight_layout()
plt.show()

## Contract Type Analysis

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# NAICS categories
naics = df.groupby('naics_desc')['award_amount'].agg(['sum','count']).sort_values('sum', ascending=False).head(10)
naics['sum'].plot(kind='barh', ax=axes[0], color='#1abc9c')
axes[0].set_title('Top 10 NAICS Categories by Obligation')
axes[0].set_xlabel('USD')

# PSC categories
psc = df.groupby('psc_desc')['award_amount'].agg(['sum','count']).sort_values('sum', ascending=False).head(10)
psc['sum'].plot(kind='barh', ax=axes[1], color='#e74c3c')
axes[1].set_title('Top 10 PSC Categories by Obligation')
axes[1].set_xlabel('USD')

plt.tight_layout()
plt.show()

## Geographic Distribution

In [ ]:
state_stats = df.groupby('state').agg({
    'award_amount': 'sum',
    'award_id': 'count'
}).sort_values('award_amount', ascending=False).head(15)

fig, ax = plt.subplots(figsize=(12, 6))
state_stats['award_amount'].plot(kind='bar', ax=ax, color='#f39c12')
ax.set_title('Top 15 States by Federal Contract Obligation')
ax.set_xlabel('State')
ax.set_ylabel('USD')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Save Enhanced Data

In [ ]:
df.to_csv('../data/contracts_schedule_enhanced.csv', index=False)
print(f"Enhanced data saved: {len(df)} contracts")